# Detección de Equipos de Protección Personal (EPP) en imágenes de obra usando YOLOv8

## Objetivo del notebook

Este notebook reproduce el flujo completo de entrenamiento, validación e inferencia de un modelo de detección de objetos utilizando YOLOv8 y un dataset personalizado alojado en Roboflow.

El objetivo del proyecto es detectar automáticamente elementos de seguridad en imágenes de obra del sector AECO (Architecture, Engineering, Construction and Operations).

## Contexto del problema

El sector AECO presenta una alta incidencia de accidentes laborales en obra. Muchos de estos accidentes se relacionan con la falta de uso de Equipos de Protección Personal (EPP), como cascos, chalecos reflectantes y botas de seguridad, además de la escasa trazabilidad de condiciones de seguridad en faena.

Este proyecto explora el uso de visión computacional para apoyar el monitoreo visual de seguridad en imágenes de obra.

## Clases utilizadas

- person
- helmet
- vest
- security_boots

## Flujo del notebook

1. Instalación de dependencias
2. Verificación del entorno de ejecución
3. Descarga del dataset desde Roboflow
4. Entrenamiento del modelo YOLOv8
5. Validación del modelo
6. Inferencia en imágenes de validación
7. Inferencia en imágenes nuevas
8. Exportación de resultados para GitHub

In [ ]:
!nvidia-smi

Sun Mar  8 11:52:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
HOME = os.getcwd()
print("Directorio actual:", HOME)

Directorio actual: /content


In [ ]:
# 1. Instalación de dependencias
#En esta sección se instalan las librerías necesarias para trabajar con YOLOv8 y Roboflow en Google Colab.

!pip uninstall -y ultralytics numpy opencv-python opencv-python-headless pandas
!pip install -q "numpy<2" "pandas<2.3" ultralytics roboflow opencv-python-headless

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-python-headless 4.13.0.92
Uninstalling opencv-python-headless-4.13.0.92:
  Successfully uninstalled opencv-python-headless-4.13.0.92
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 7.5 MB/s eta 0:00:00
ERROR: Operation cancelled by user
^C


In [ ]:
# 2. Verificación del entorno
# Se verifica si Google Colab dispone de GPU para acelerar el entrenamiento.

import numpy as np
import pandas as pd
import torch
import ultralytics

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Torch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ModuleNotFoundError: No module named 'pandas'

In [ ]:
from ultralytics import YOLO
from roboflow import Roboflow
from IPython.display import display, Image

import yaml
import glob
import shutil

In [ ]:
# 3. Descarga del dataset desde Roboflow
#En esta sección se descarga la versión del dataset utilizada para el proyecto.
!mkdir -p {HOME}/datasets
%cd {HOME}/datasets

In [ ]:
rf = Roboflow(api_key="A9f5IpNZpWSut5QaBAMB")

project = rf.workspace("domenicas-workspace").project("m4t3")

dataset = project.version(4).download("yolov8")

In [ ]:
# Verificar el Dataset

DATASET_DIR = dataset.location
DATA_YAML = os.path.join(DATASET_DIR, "data.yaml")

print("Dataset location:", DATASET_DIR)

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

print(data_cfg)

In [ ]:
# Volver al directorio principal

%cd {HOME}

In [ ]:
# 4. Entrenamiento del modelo YOLOv8
#Se entrena el modelo con los parámetros definidos para este experimento.

model = YOLO("yolov8s.pt")

train_results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=16,
    project=f"{HOME}/runs_aeco",
    name="ppe_yolov8s_exp1",
    pretrained=True,
    patience=20,
    plots=True
)

In [ ]:
#Revisar resultados del entrenamiento

RUN_DIR = f"{HOME}/runs_aeco/ppe_yolov8s_exp1"

print(os.listdir(RUN_DIR))

In [ ]:
# Mostrar las curvas

curve_files = [
    "results.png",
    "confusion_matrix.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png"
]

for file in curve_files:
    path = os.path.join(RUN_DIR, file)
    if os.path.exists(path):
        display(Image(filename=path))

In [ ]:
#Validar el modelo

best_weights = os.path.join(RUN_DIR, "weights", "best.pt")

best_model = YOLO(best_weights)

metrics = best_model.val(data=DATA_YAML)

print(metrics)

In [ ]:
# Inferencia en imágenes de validación

valid_images = glob.glob(f"{DATASET_DIR}/valid/images/*.*")

print(len(valid_images))
print(valid_images[:5])

# Predicción

val_results = best_model.predict(
    source=valid_images[:10],
    conf=0.25,
    save=True,
    project=f"{HOME}/inference_outputs",
    name="validation_preds",
    exist_ok=True
)

# Mostrar predicciones

val_pred_dir = f"{HOME}/inference_outputs/validation_preds"

pred_imgs = glob.glob(f"{val_pred_dir}/*.*")

for img_path in pred_imgs[:10]:
    display(Image(filename=img_path, width=600))

In [ ]:
# Inferencia en imágenes nuevas

#Subier imagenes
from google.colab import files
uploaded = files.upload()

In [ ]:
NEW_IMG_DIR = f"{HOME}/new_images"
os.makedirs(NEW_IMG_DIR, exist_ok=True)

for file_name in uploaded.keys():
    shutil.move(file_name, os.path.join(NEW_IMG_DIR, file_name))

print("Imágenes nuevas guardadas en:", NEW_IMG_DIR)
print("Contenido:", os.listdir(NEW_IMG_DIR))

In [ ]:
# Mostrar las predicciones de las imágenes nuevas

new_pred_dir = f"{HOME}/inference_outputs/new_images_preds"
pred_new_imgs = glob.glob(f"{new_pred_dir}/*.*")

print("Cantidad de imágenes predichas:", len(pred_new_imgs))

for img_path in pred_new_imgs:
    display(Image(filename=img_path, width=600))

In [ ]:
print("HOME:", HOME)
print("NEW_IMG_DIR:", NEW_IMG_DIR)

print("Contenido de NEW_IMG_DIR:")
print(os.listdir(NEW_IMG_DIR))

In [ ]:
import glob

new_image_files = glob.glob(f"{NEW_IMG_DIR}/*.*")

print("Imágenes nuevas detectadas:")
print(new_image_files)

In [ ]:
new_results = best_model.predict(
    source=new_image_files,
    conf=0.25,
    save=True,
    project=f"{HOME}/inference_outputs",
    name="new_images_preds",
    exist_ok=True
)

print("Cantidad de resultados devueltos:", len(new_results))

In [ ]:
pred_new_imgs = glob.glob(f"{HOME}/inference_outputs/**/*.*", recursive=True)

print("Archivos encontrados:")
for p in pred_new_imgs:
    print(p)

In [ ]:
from IPython.display import display, Image

for img_path in pred_new_imgs:
    if img_path.lower().endswith((".jpg", ".jpeg", ".png")):
        display(Image(filename=img_path, width=600))

In [ ]:
print("Carpeta del experimento:", RUN_DIR)
print(os.listdir(RUN_DIR))

In [ ]:
import glob
from IPython.display import Image, display

annot_examples = glob.glob(f"{DATASET_DIR}/train/images/*")[:5]

print("Ejemplos de imágenes del dataset:")

for img in annot_examples:
    display(Image(filename=img, width=500))

In [ ]:
val_pred_dir = f"{HOME}/inference_outputs/validation_preds"

pred_imgs = glob.glob(f"{val_pred_dir}/*")

for img in pred_imgs[:10]:
    display(Image(filename=img, width=600))

In [ ]:
import os

HOME = os.getcwd()
print("Directorio actual:", HOME)

In [ ]:
import glob
from IPython.display import display, Image

In [ ]:
val_pred_dir = f"{HOME}/inference_outputs/validation_preds"

print("Carpeta:", val_pred_dir)

In [ ]:
pred_imgs = glob.glob(f"{val_pred_dir}/*")

print("Cantidad de imágenes encontradas:", len(pred_imgs))